In [1]:

import tensorflow as tf
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

# 1. Configuration
DATA_DIR = "data/flower_photos"
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
SEED = 42

# 2. Load training images (80%)
train_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    validation_split=0.2,
    subset="training",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

# Load validation images (20%)
val_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    validation_split=0.2,
    subset="validation",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

class_names = train_ds.class_names
num_classes = len(class_names)

print("Classes:", class_names)

# 3. Optimize input pipeline
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(AUTOTUNE)
val_ds = val_ds.prefetch(AUTOTUNE)

# 4. Load pretrained MobileNetV2
base_model = MobileNetV2(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3),
    pooling="avg"
)

# PHASE 1: Freeze the pretrained backbone
base_model.trainable = False

inputs = layers.Input(shape=(224, 224, 3))
x = layers.Lambda(preprocess_input)(inputs)
x = base_model(x, training=False)
x = layers.Dropout(0.2)(x)
outputs = layers.Dense(num_classes, activation="softmax")(x)

model = Model(inputs, outputs)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

print("\nPhase 1: Training classification head")
model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=5
)

# PHASE 2: Fine-tune the upper MobileNetV2 layers
base_model.trainable = True

# Freeze the earlier layers; unfreeze only the last 30
for layer in base_model.layers[:-30]:
    layer.trainable = False

# Keep BatchNormalization layers frozen for stability
for layer in base_model.layers:
    if isinstance(layer, layers.BatchNormalization):
        layer.trainable = False

# Recompile with a very low learning rate
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

print("\nPhase 2: Fine-tuning MobileNetV2")
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=3,
            restore_best_weights=True
        ),
        tf.keras.callbacks.ModelCheckpoint(
            "best_flower_mobilenet.keras",
            monitor="val_loss",
            save_best_only=True
        )
    ]
)



# 6. Display class names and final validation performance
loss, accuracy = model.evaluate(val_ds)
print("Validation accuracy:", accuracy)
print("Classes:", class_names)


Found 3670 files belonging to 5 classes.
Using 2936 files for training.
Found 3670 files belonging to 5 classes.
Using 734 files for validation.
Classes: ['daisy', 'dandelion', 'roses', 'sunflowers', 'tulips']


Phase 1: Training classification head
Epoch 1/5


c:\Users\windows\Desktop\projects\computer-vision\.venv\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


92/92 ━━━━━━━━━━━━━━━━━━━━ 101s 994ms/step - accuracy: 0.7006 - loss: 0.8131 - val_accuracy: 0.8501 - val_loss: 0.4659
Epoch 2/5
92/92 ━━━━━━━━━━━━━━━━━━━━ 92s 993ms/step - accuracy: 0.8573 - loss: 0.4068 - val_accuracy: 0.8719 - val_loss: 0.3816
Epoch 3/5
92/92 ━━━━━━━━━━━━━━━━━━━━ 106s 602ms/step - accuracy: 0.8873 - loss: 0.3278 - val_accuracy: 0.8719 - val_loss: 0.3915
Epoch 4/5
92/92 ━━━━━━━━━━━━━━━━━━━━ 54s 596ms/step - accuracy: 0.9084 - loss: 0.2746 - val_accuracy: 0.8760 - val_loss: 0.3736
Epoch 5/5
92/92 ━━━━━━━━━━━━━━━━━━━━ 96s 741ms/step - accuracy: 0.9237 - loss: 0.2490 - val_accuracy: 0.8910 - val_loss: 0.3405

Phase 2: Fine-tuning MobileNetV2
Epoch 1/10
92/92 ━━━━━━━━━━━━━━━━━━━━ 34s 325ms/step - accuracy: 0.9373 - loss: 0.1969 - val_accuracy: 0.9019 - val_loss: 0.3211
Epoch 2/10
92/92 ━━━━━━━━━━━━━━━━━━━━ 55s 605ms/step - accuracy: 0.9458 - loss: 0.1549 - val_accuracy: 0.9101 - val_loss: 0.3024
Epoch 3/10
92/92 ━━━━━━━━━━━━━━━━━━━━ 73s 799ms/step - accuracy: 0.9591 - lo